<a href="https://colab.research.google.com/github/tanishkadawake/all-project/blob/main/nlpexam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#design a program to analyze discourse structure in a paragraph . implement a sample reference resoulution system to identify pronouns and their corresponding noun phrases.. analyze discourse coherence using sample text

# Discourse Structure & Reference Resolution Analyzer

This notebook contains:
1. **Pronoun Reference Resolution**: A rule-based system matching pronouns to the most likely preceding noun phrase based on gender/number heuristics.
2. **Discourse Coherence Analysis**: Evaluation of coherence based on entity overlap (centering theory concepts) and sentence similarity.

In [ ]:
#Tanishka_Dawake_RBT23CB030

In [ ]:
import spacy
from collections import Counter

# Load English pipeline
nlp = spacy.load("en_core_web_sm")

In [ ]:
def resolve_pronouns(text):
    doc = nlp(text)
    resolved_story = []

    # Mapping rules for simple gender/number pronouns
    pronouns_to_resolve = {"he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their"}

    # Track candidate noun phrases found so far
    candidates = []

    for sent in doc.sents:
        resolved_sent_tokens = []

        # Gather noun chunks in this sentence to act as future candidates
        for chunk in sent.noun_chunks:
            # Filter out simple pronouns from candidate list
            if chunk.text.lower() not in pronouns_to_resolve:
                candidates.append(chunk)

        for token in sent:
            token_text = token.text
            lower_token = token.text.lower()

            if lower_token in pronouns_to_resolve:
                # Basic heuristic: resolve to the most recent noun chunk
                if candidates:
                    # Simplistic match: choose the closest preceding noun chunk
                    best_candidate = candidates[-1]
                    resolved_text = f"{token_text} [{best_candidate.text}]"
                    resolved_sent_tokens.append(resolved_text)
                else:
                    resolved_sent_tokens.append(token_text)
            else:
                resolved_sent_tokens.append(token_text)

        resolved_story.append(" ".join(resolved_sent_tokens))

    return " ".join(resolved_story)

In [ ]:
def analyze_coherence(text):
    doc = nlp(text)
    sentences = list(doc.sents)

    if len(sentences) < 2:
        return "Text is too short to compute discourse coherence transitions."

    coherence_report = []

    # Entity Grid / Overlap check between consecutive sentences
    for i in range(len(sentences) - 1):
        sent1 = sentences[i]
        sent2 = sentences[i+1]

        nouns1 = {token.lemma_.lower() for token in sent1 if token.pos_ in {"NOUN", "PROPN"}}
        nouns2 = {token.lemma_.lower() for token in sent2 if token.pos_ in {"NOUN", "PROPN"}}

        overlap = nouns1.intersection(nouns2)
        similarity = sent1.similarity(sent2)

        coherence_report.append({
            "transition": f"Sentence {i+1} -> Sentence {i+2}",
            "overlap_entities": list(overlap),
            "similarity_score": round(similarity, 4)
        })

    return coherence_report

In [ ]:
# Sample Discourse Paragraph
sample_paragraph = (
    "Alice went to the garden. She found a beautiful rose there. "
    "The rose was bright red. It smelled wonderful. "
    "Bob joined her later. He brought some water for the plants."
)

print("=== Original Paragraph ===")
print(sample_paragraph)
print("\n=== Pronoun Resolution Demo ===")
print(resolve_pronouns(sample_paragraph))

print("\n=== Discourse Coherence Analysis ===")
coherence = analyze_coherence(sample_paragraph)
for step in coherence:
    print(f"{step['transition']}:")
    print(f"  - Overlapping Entities: {step['overlap_entities']}")
    print(f"  - Sentence Similarity: {step['similarity_score']}")

=== Original Paragraph ===
Alice went to the garden. She found a beautiful rose there. The rose was bright red. It smelled wonderful. Bob joined her later. He brought some water for the plants.

=== Pronoun Resolution Demo ===
Alice went to the garden . She [a beautiful rose] found a beautiful rose there . The rose was bright red . It [The rose] smelled wonderful . Bob joined her [Bob] later . He [the plants] brought some water for the plants .

=== Discourse Coherence Analysis ===
Sentence 1 -> Sentence 2:
  - Overlapping Entities: []
  - Sentence Similarity: 0.5706
Sentence 2 -> Sentence 3:
  - Overlapping Entities: ['rose']
  - Sentence Similarity: 0.6928
Sentence 3 -> Sentence 4:
  - Overlapping Entities: []
  - Sentence Similarity: 0.5952
Sentence 4 -> Sentence 5:
  - Overlapping Entities: []
  - Sentence Similarity: 0.6603
Sentence 5 -> Sentence 6:
  - Overlapping Entities: []
  - Sentence Similarity: 0.3478


/tmp/ipykernel_1364/3824572350.py:19: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Span.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  similarity = sent1.similarity(sent2)
